# Collector Blackouts (Polars)

Estimates data lost during collector poll gaps using the Polars analysis path.

In [1]:
from pathlib import Path
import importlib.util
import sys

import plotly.express as px
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "analysis" / "polars").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the project root or a subdirectory.")

POLARS_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "polars"
polars_analysis_path = str(POLARS_ANALYSIS_DIR)
if polars_analysis_path in sys.path:
    sys.path.remove(polars_analysis_path)
sys.path.insert(0, polars_analysis_path)

for module_name in ("_shared", "report_cache", "cli_common"):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None) if module else None
    module_path = Path(module_file).resolve() if module_file else None
    if module_path is not None and POLARS_ANALYSIS_DIR not in module_path.parents:
        sys.modules.pop(module_name, None)

def load_polars_script(module_name: str, filename: str):
    spec = importlib.util.spec_from_file_location(module_name, POLARS_ANALYSIS_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

collector_blackouts = load_polars_script("polars_collector_blackouts", "collector-blackouts.py")

DB = PROJECT_ROOT / "data" / "foli.db"
CACHE_DIR = PROJECT_ROOT / "outputs" / "polars-report-cache"
FORCE_CACHE = False
NO_CACHE = False
LIMIT = 20
MIN_OBSERVATIONS = 1


In [2]:
settings = collector_blackouts.ReportSettings(
    db=DB,
    limit=LIMIT,
    min_observations=MIN_OBSERVATIONS,
).resolved()
polls = collector_blackouts.load_collector_polls(settings)
summary = collector_blackouts.build_collector_blackouts(polls, LIMIT)
summary


source,poll_count,success_count,failed_count,expected_cadence_seconds,blackout_count,total_blackout_min,largest_blackout_min,estimated_missed_polls,estimated_missed_rows
str,i64,i64,i64,f64,i64,f64,f64,f64,f64
"""siri_vm""",42556,42454,102,30.0,25,216.22,47.77,432.43,57521.45
"""siri_alerts""",4310,4297,13,300.0,3,55.43,34.15,11.09,144.57
"""gtfs""",3,3,0,604808.0,0,0.0,0.0,0.0,0.0


In [3]:
if summary.is_empty():
    print("No collector polls found.")
else:
    fig = px.bar(
        summary.sort("total_blackout_min"),
        x="total_blackout_min",
        y="source",
        orientation="h",
        title="Estimated blackout duration by source",
        labels={"source": "Source", "total_blackout_min": "Estimated blackout minutes"},
    )
    fig.update_layout(showlegend=False)
    fig.show()
